# Análise interseccional da desigualdade

Este notebook consolida uma leitura descritiva dos marcadores combinados de vulnerabilidade nos casos notificados de sífilis congênita em Porto Alegre, RS, Brasil.

A análise cruza grupo racial materno, escolaridade, pré-natal, momento do diagnóstico e tratamento materno registrado. O objetivo não é inferir causalidade, mas identificar se marcadores de maior vulnerabilidade aparecem de forma desigual entre os grupos.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

from src.config import DEFAULT_DATABASE_URL, ROOT, load_project_env
from src.visualization.final_report import generate_all

load_project_env()
DATABASE_URL = os.getenv("DATABASE_URL", DEFAULT_DATABASE_URL)
engine = create_engine(DATABASE_URL)

QUERY_DIR = ROOT / "database" / "queries"


## Incidência por raça/cor detalhada

A visualização detalhada por raça/cor é complementar. A comparação principal do projeto permanece agrupada em mães negras e mães não negras. Categorias com baixo denominador são mantidas na consulta e sinalizadas com cautela metodológica.


In [ ]:
sql_raca_detalhada = (QUERY_DIR / "16_incidencia_raca_cor_detalhada.sql").read_text(encoding="utf-8")
incidencia_detalhada = pd.read_sql_query(text(sql_raca_detalhada), engine)
display(incidencia_detalhada.head(20))


## Marcador interseccional de maior vulnerabilidade

O marcador combina baixa ou ignorada escolaridade com pelo menos um registro de cuidado ausente, tardio, inadequado ou ignorado. A leitura é descritiva e deve ser usada como sinal de concentração de vulnerabilidades, não como prova causal.


In [ ]:
sql_interseccional = (QUERY_DIR / "17_analise_interseccional_desigualdade.sql").read_text(encoding="utf-8")
interseccional = pd.read_sql_query(text(sql_interseccional), engine)
display(interseccional)


## Perfil de mães negras por escolaridade e idade

Este recorte ajuda a qualificar o perfil dos casos entre mães negras, usando o campo `ANT_IDADE` do SINAN/SIFCBR para faixa etária materna e o agrupamento analítico de escolaridade já validado no projeto.


In [ ]:
sql_perfil = (QUERY_DIR / "18_perfil_maes_negras_escolaridade_idade.sql").read_text(encoding="utf-8")
perfil_maes_negras = pd.read_sql_query(text(sql_perfil), engine)
display(perfil_maes_negras)


## Exportação de imagens

As imagens finais são exportadas para `outputs/images/final_report/`.


In [ ]:
imagens = generate_all(DATABASE_URL, "outputs/images/final_report")
for imagem in imagens:
    print(imagem.relative_to(ROOT))


## Leitura técnica

A análise interseccional reforça a leitura central do projeto: a desigualdade mais consistente aparece na incidência por grupo racial ao longo da série histórica, e os marcadores de cuidado e perfil socioeconômico ajudam a qualificar essa desigualdade. As categorias detalhadas com baixo denominador devem ser citadas como complemento exploratório, não como evidência principal.
